# 🧊🔥 Mutability vs Immutability — Whiteboard or Stone Tablet?

**Folder:** 00_Python → 03_Pythonic_Thinking → 04_Mutability_vs_Immutability
**Level:** 🐔 Intermediate · **Type:** THEORY (id() X-ray lab 🔬)

---

Everything you store in Python is an **object** that lives at some address in
memory, and the built-in `id()` function hands you that address. Mutability is
simply the answer to one question: when the value changes, does the object stay
at the same address, or does Python build a fresh object somewhere else?

- A **mutable** object (`list`, `dict`, `set`, `bytearray`) can be edited
  **in place** — the same `id()` before and after the edit. Think of a whiteboard:
  you erase and rewrite on the very same board.
- An **immutable** object (`int`, `float`, `bool`, `str`, `tuple`, `frozenset`,
  `bytes`, `None`) can never be edited. Every "change" creates a **new object**
  with a **new `id()`**. Think of a stone tablet: to write something new you must
  carve a brand-new tablet.

Why does this matter in real code? Mutability is the root of two of the most
famous Python surprises: two names pointing at the **same** list and changing it
"by accident" (aliasing), and the legendary mutable-default-argument trap inside
functions. It also decides what may become a dictionary key or a set member, and
it explains why `list += ...` behaves differently from `tuple += ...`.

## 🎯 Roadmap for this notebook
1. The two clubs — mutable vs immutable — and the lunchbox story that fixes it in memory.
2. `id()` X-ray #1 — numbers: arithmetic always produces a new object, plus the small-int interning bonus.
3. `id()` X-ray #2 — strings: concatenation and methods never touch the original.
4. Lists versus tuples — in-place mutation and the `+=` double standard.
5. Hashability — why only frozen objects are allowed as dict keys or set members.
6. Identity versus equality — `is` and `==` answer two different questions.

**One-liner:**
> Mutable = a **whiteboard** 🧽 (erase and rewrite on the same one). Immutable =
> a **stone tablet** 🪨 (any new writing needs a brand-new tablet).


## 🌍 Real-World Hook: The 2 Brothers of Lunchbox-Service 🍱

- **Mutable Max** 🔥 = cottage cheese today, mixed vegetables tomorrow — the SAME
  lunchbox, only the curry changed (in-place, same id!).
- **Immutable Ivan** 🧊 = he will bring a NEW lunchbox for any new curry —
  every change = a new container (a new id)!

The Python family sorted:

| 🔥 MUTABLE (changeable, same id) | 🧊 IMMUTABLE (a new object every change) |
|---|---|
| `list` | `int`, `float`, `bool` |
| `dict` | `str` |
| `set` | `tuple`, `frozenset` |
| `bytearray` | `bytes`, `NoneType` |

### 📖 Definitions
> **Mutable:** the content can change IN-PLACE (object/address stays SAME) 🔥
> **Immutable:** to change the content a NEW object is created (address changes!) 🧊


# 1. The id() X-ray — Watching Immutability Live

## 1.1 Numbers: Every Arithmetic Step Builds a New Object

`id()` returns the memory address of an object as an integer. Because numbers
are immutable, `x = x + 1` does **not** modify the object `100`; it creates a
new object `101` and rebinds the name `x` to it. The old `100` may still exist
elsewhere, completely untouched.

A useful mental model: the name is a label, the object is a box. For immutable
types the label keeps jumping to a new box; for mutable types the label stays
glued to the same box and only the inside changes.

The second half of the experiment reveals **small-integer interning**, a CPython
optimisation: the integers `-5` through `256` are pre-created and reused, so two
assignments to `5` often point at the *same* object. Larger values such as
`1000` are freshly allocated each time, so two equal values live at different
addresses.

| Expression | Result | Why |
|---|---|---|
| `x = 100; x = x + 1` | new `id()` | arithmetic builds a new int |
| `int("5") is int("5")` | usually `True` | small ints are cached |
| `int("1000") is int("1000")` | `False` | big ints are fresh objects |

> ⚠️ Interning is an implementation detail. Compare values with `==`, never with `is`.


In [1]:
# 🐣 id() X-ray LESSON #1 — numbers hamesha naya object! 🧊
x = 100
print("x pehle  :", x, "| id:", id(x))
x = x + 1                              # math kiya → NAYA object bana
print("x baad   :", x, "| id:", id(x), "← ID BADAL GAYA! 🧊 (imm)")

# ✨ BONUS: chhote ints interned hote — id kabh-kabhi SAME dikhta (CPython jugaad!)
a, b = int("5"), int("5")              # int("...") se FRESH parse — folding nahi!
print("\n5 vs 5 (small-int intern):", a is b, "← chhote ints cached details 😇")
a, b = int("1000"), int("1000")
print("1000 vs 1000:", a is b, "← bade ints fresh objects — dono same-value, different-identity!")
# Expected OUTPUT:
# x pehle  : 100 | id: <n1>
# x baad   : 101 | id: <n2> ← ID BADAL GAYA! 🧊 (imm)
#
# 5 vs 5 (small-int intern): True ← chhote ints cached details 😇
# 1000 vs 1000: False ← bade ints fresh objects — dono same-value, different-identity!

x pehle  : 100 | id: 140726039720152
x baad   : 101 | id: 140726039720184 ← ID BADAL GAYA! 🧊 (imm)

5 vs 5 (small-int intern): True ← chhote ints cached details 😇
1000 vs 1000: False ← bade ints fresh objects — dono same-value, different-identity!


## 1.2 Strings: Concatenation and Methods Return New Objects

A string is immutable, so `s = s + " pila dijiye"` builds a brand-new string
object and rebinds `s`. The original object stays frozen, which is why building a
string inside a loop is expensive: every `+` allocates and copies the whole
result.

Exactly the same rule applies to string **methods**: `s.upper()`, `s.strip()`
and `s.replace(...)` return **new** strings and never modify the receiver. A
method that appears to "change" a string is really handing you a new object.

```text
s = "chai"              # object A at address id1
s = s + " pila dijiye"  # object B at address id2, A still frozen
s.upper()               # object C, s still points to B
```

This is a safety feature: because a string can be shared freely between many
parts of a program, no function can accidentally corrupt it. It is also exactly
why strings are legal dictionary keys and set members — a rule you will meet
again in section 3.


In [2]:
# 🐣 id() X-ray LESSON #2 — strings ka tablet-roe 🧊
s = "chai"
print("s pehle :", s, "| id:", id(s))
s = s + " pila dijiye"
print("s baad  :", s, "| id:", id(s), "← concat pe NAYA string bana 🧊")

# String METHOD bhi nayata deta — OG never change
big = s.upper()
print("\nmethod result:", big)
print("OG unchanged   :", s, "← immutability OP 😎")
# Expected OUTPUT:
# s pehle : chai | id: <n1>
# s baad  : chai pila dijiye | id: <n2> ← concat pe NAYA string bana 🧊
#
# method result: CHAI PILA DIJIYE
# OG unchanged   : chai pila dijiye ← immutability OP 😎

s pehle : chai | id: 2049302561488
s baad  : chai pila dijiye | id: 2049347281392 ← concat pe NAYA string bana 🧊

method result: CHAI PILA DIJIYE
OG unchanged   : chai pila dijiye ← immutability OP 😎


# 2. Mutable Containers and the `+=` Double Standard

## 2.1 Lists Change In Place; Tuples Are Rebuilt

A list is mutable: `append`, `extend`, `pop`, `sort` and item assignment all
change the object **in place**, so its `id()` is unchanged. The statement
`lst += [100]` is the interesting case. Python calls the in-place operator
`__iadd__` on the list, which extends the very same object, so the `id()` stays
the same. That is efficient, but dangerous when another name points at the same
list (aliasing): both names observe the change.

A tuple is immutable, so `t += (4,)` has no in-place operation to call. Python
falls back to `t = t + (4,)`, which creates a **new** tuple and rebinds `t`. The
address changes, and any other name still points at the old, unchanged tuple.

| Operation | Type | In place? | `id()` |
|---|---|---|---|
| `lst.append(x)` | `list` | yes | same |
| `lst += [x]` | `list` | yes | same |
| `t += (x,)` | `tuple` | no | new |
| `s += "x"` | `str` | no | new |

> 🎯 Rule of thumb: for mutable containers `+=` mutates; for immutable types it
> rebinds. Look at the object's `id()` whenever you are unsure.


In [3]:
# 🐥 id() X-ray LESSON #3 — LIST: whiteboard live! 🔥
lst = [1, 2, 3]
print("lst pehle   :", lst, "| id:", id(lst))
lst.append(99)
print("append ke baad:", lst, "| id:", id(lst), "← id SAME! in-place modify 🔥")
lst += [100]                           # += LIST PE BHI IN-PLACE!
print("lst += [100]   :", lst, "| id:", id(lst), "← STILL SAME!")

# TUPLE pe += ?? REBUILD hota hai (imm yaad!) 🧊
t = (1, 2, 3)
id_before = id(t)
t = t + (4,)
print("\ntuple += ke baad:", t, "| id changed?", id(t) != id_before, "← REBUILD hua 😲")
# Expected OUTPUT:
# lst pehle   : [1, 2, 3] | id: <n1>
# append ke baad: [1, 2, 3, 99] | id: <n1> ← id SAME! in-place modify 🔥
# lst += [100]   : [1, 2, 3, 99, 100] | id: <n1> ← STILL SAME!
#
# tuple += ke baad: (1, 2, 3, 4) | id changed? True ← REBUILD hua 😲

lst pehle   : [1, 2, 3] | id: 2049347068352
append ke baad: [1, 2, 3, 99] | id: 2049347068352 ← id SAME! in-place modify 🔥
lst += [100]   : [1, 2, 3, 99, 100] | id: 2049347068352 ← STILL SAME!

tuple += ke baad: (1, 2, 3, 4) | id changed? True ← REBUILD hua 😲


# 3. Hashability — The Members-Only Rule

## 3.1 Only Frozen (Immutable) Objects May Be Keys

A Python `dict` key and a `set` member must be **hashable**. Hashing turns a
value into a number that the table uses for fast lookups, and that number must
never change while the object sits inside the container. A mutable object could
change after insertion and silently corrupt the lookup table, so Python forbids
it.

Consequently:

- **hashable**: `str`, `int`, `float`, `bool`, `tuple` (only if every element is
  itself hashable), `frozenset`, `None`.
- **not hashable**: `list`, `dict`, `set`, `bytearray`.

A subtle trap is a **tuple that contains a list**: the tuple looks immutable, but
one of its elements is mutable, so the whole thing is unhashable and cannot be a
key. Use a nested tuple or a `frozenset` instead. The practical summary is that
hashability tracks immutability closely, and `hash(obj)` raises
`TypeError: unhashable type` the moment the rule is broken.


In [4]:
# 🐔 HASH-KEY rule — dict ka members-only club! 🎫
phone = {"Ravi": "98765", "Meena": "91234"}        # str keys (imm) ✅
print("str keys work:", phone)

bad_key = (["a", "b"], 2)                           # list andar — unhashable!
try:
    hash(bad_key)
except TypeError as e:
    print("tuple-with-list 🚨:", e, "← dict-key NAKAAM!")

# Fix: immutable versions use karo!
good = (("a", "b"), 2)
print("pure-tuple hash OK:", hash(good) is not None, "→ dict-key valid ✅")
boot = {[1, 2]: "nope"} if False else "sets pe hash nahi — frozenset YES ✅"
print(boot)
# Expected OUTPUT:
# str keys work: {'Ravi': '98765', 'Meena': '91234'}
# tuple-with-list 🚨: unhashable type: 'list' ← dict-key NAKAAM!
# pure-tuple hash OK: True → dict-key valid ✅
# sets pe hash nahi — frozenset YES ✅

str keys work: {'Ravi': '98765', 'Meena': '91234'}
tuple-with-list 🚨: unhashable type: 'list' ← dict-key NAKAAM!
pure-tuple hash OK: True → dict-key valid ✅
sets pe hash nahi — frozenset YES ✅


# 4. Identity vs Equality — `is` versus `==`

## 4.1 Two Names, One Object — Aliasing and the Meaning of `is`

Python gives you two comparison operators that answer two different questions:

- `==` asks **"are these two values equal?"** It calls `__eq__` and compares
  content, so `[1, 2, 3] == [1, 2, 3]` is `True`.
- `is` asks **"are these literally the same object?"** It compares `id()`, so
  `[1, 2, 3] is [1, 2, 3]` is `False` because two separate lists were built.

Assignment never copies: `z = x_list` makes `z` a second label for the *same*
list, so `z is x_list` is `True`. Editing through either name changes both, which
is the aliasing behaviour that bites real programs.

Use `is` in exactly two everyday situations: comparing against `None`
(`if x is None`) and checking a sentinel object. For everything else — numbers,
strings, containers — use `==`. Small-int and string interning can make `is`
accidentally return `True`, so relying on it for value comparison produces code
that breaks as soon as the data gets larger.


In [5]:
# 🐣 is vs == ka pehla jhala — same-value ≠ same-object! 👯
x_list = [1, 2, 3]
y_list = [1, 2, 3]
print("x == y:", x_list == y_list, "← VALUES same ✅")
print("x is y:", x_list is y_list, "← OBJECTS alag (do dabbas, same subzi!)")

# is = identity-train (id matching), == = equality-punch (values matching)
z = x_list
print("\nz = x_list; z is x:", z is x_list, "← aliasing — ab SAME dabba hai 🏷️🏷️")
# Expected OUTPUT:
# x == y: True ← VALUES same ✅
# x is y: False ← OBJECTS alag (do dabbas, same subzi!)
#
# z = x_list; z is x: True ← aliasing — ab SAME dabba hai 🏷️🏷️

x == y: True ← VALUES same ✅
x is y: False ← OBJECTS alag (do dabbas, same subzi!)

z = x_list; z is x: True ← aliasing — ab SAME dabba hai 🏷️🏷️


## ⚠️ 5 Common Mistakes (Family Special)

| # | ❌ Mistake | ✅ Correct |
|---|---|---|
| 1 | `x is y` for value-comparisons | `x == y`; use `is` only for identity/None-checks |
| 2 | Expecting in-place on `+=` for every type | list-in-place 🔥, tuple/num/str REBUILD 🧊 |
| 3 | A list/tuple-with-list in a dict-key | Only immutables (str/num/pure-tuple/frozenset) |
| 4 | Business-logic on interning (`is`) | A CPython implementation-detail — always compare with `==` |
| 5 | "Tuple immutable, so safe" in nested cases | A tuple-with-a-mutable-list = mutability leakage! 🕳️ |

> 🎯 **Feeling:** id() = the object's aadhaar-card. In mutables the aadhaar stays the same,
> only the inner-city changes; in immutables a change = a fresh aadhaar! 💳

## Summary

Mutability decides whether an object can be changed in place. Mutable objects —
`list`, `dict`, `set`, `bytearray` — keep the same `id()` while their contents
change, so every name pointing at them sees the edit. Immutable objects —
numbers, strings, tuples, `frozenset`, `bytes` — are never modified; any "change"
rebinds the name to a new object with a new `id()`.

The `id()` X-ray proved this for numbers, strings, lists and tuples, and showed
that `+=` mutates a list but rebuilds a tuple. Hashability follows immutability,
so only frozen values may be dictionary keys or set members. Finally, `==`
compares values while `is` compares identity, and plain assignment creates an
alias, not a copy.

### ⏭️ Next up
`02_memory_behavior_experiments.ipynb` — the mutable-default-argument trap,
caller-side side effects, and the cost of join versus concatenation.
